In [1]:
# 07_economic_impact.ipynb
#
# Experiment 7 -- Economic impact of the fairness conditions (finance dimension).
#
# Purpose:
#   Translate the fairness conditions into lender economics. Following the
#   profit-oriented view of credit scoring, each approved non-defaulter earns a
#   return r on the exposure and each approved defaulter loses a fraction l; the
#   lender's profit per applicant is a simple function of the approval rule.
#   We report, per condition and dataset: overall approval rate, the approval
#   rates of the disadvantaged and reference groups, and normalized profit, so
#   that the fairness-utility-legality trade-off also carries a monetary and an
#   access (approval-volume) reading. Protected attributes: German=age,
#   Taiwan=sex, GMSC=age. 30 seeds, 95% CIs.
#
# Simple profit model (stylized, unit exposure):
#   profit = (#approved non-default) * r  -  (#approved default) * l,  normalized per applicant.
#   Defaults use r=0.25 (return on a good loan) and l=1.0 (loss given default),
#   a conservative asymmetry standard in credit-scoring profit studies.
#
# Outputs (PNG+PDF, 600 dpi, grayscale, no captions; paths not printed):
#   results/tables/exp7_economics.csv
#   results/figures/exp7_approval_profit.(png|pdf)

import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
FIG_DIR  = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR  = os.path.join(PROJECT_ROOT, "results", "tables")
for _d in (DATA_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(_d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85"})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

R_RETURN, L_LOSS = 0.25, 1.0  # return on good loan; loss given default

def economics(y_true, y_pred, group):
    approved = (y_pred == 0)                 # favorable = approve = predicted non-default
    good = approved & (y_true == 0)
    bad  = approved & (y_true == 1)
    profit = (good.sum()*R_RETURN - bad.sum()*L_LOSS) / len(y_true)
    appr_overall = approved.mean()
    a1 = approved[group==1].mean(); a0 = approved[group==0].mean()
    return appr_overall, a1, a0, profit

def load_german():
    df=pd.read_csv(os.path.join(DATA_DIR,"german_credit.csv"))
    y=(df.credit_risk==0).astype(int).values; g=(df.age<25).astype(int).values
    cat=[c for c in df.select_dtypes(include=['object','string']).columns]
    X=pd.get_dummies(df.drop(columns=['credit_risk','age']),columns=cat).astype(float).reset_index(drop=True)
    return X,y,g,None
def load_taiwan():
    df=pd.read_parquet(os.path.join(DATA_DIR,"taiwan_credit.parquet"))
    y=df.DEFAULT.astype(int).values; g=(df.SEX==2).astype(int).values
    X=df.drop(columns=[c for c in ['DEFAULT','VIP','SEX_STD','AGE_YEARS','SEX','AGE','MARRIAGE'] if c in df.columns]).astype(float).reset_index(drop=True)
    return X,y,g,None
def load_gmsc():
    df=pd.read_parquet(os.path.join(DATA_DIR,"gmsc_credit.parquet"))
    y=df.DEFAULT.astype(int).values; g=(df.AGE_YEARS<40).astype(int).values
    X=df.drop(columns=[c for c in ['DEFAULT','VIP','AGE_YEARS'] if c in df.columns]).astype(float).reset_index(drop=True)
    X=X.fillna(X.median(numeric_only=True)); return X,y,g,20000

def run(X,y,g,seed,cond,subs):
    idx=np.arange(len(y))
    if subs and len(y)>subs: idx=np.random.default_rng(seed).choice(idx,subs,replace=False)
    Xs=X.iloc[idx].reset_index(drop=True); ys=y[idx]; gs=g[idx]
    Xtr,Xte,ytr,yte,gtr,gte=train_test_split(Xs,ys,gs,test_size=0.3,random_state=seed,stratify=ys)
    sc=StandardScaler().fit(Xtr); m=LogisticRegression(max_iter=2000).fit(sc.transform(Xtr),ytr)
    p=m.predict_proba(sc.transform(Xte))[:,1]
    if cond=="blind":
        yp=(p>=0.5).astype(int)
    else:  # gov_engine group-wise calibration
        br=((p>=0.5).astype(int)==0).mean()
        t1=np.quantile(p[gte==1],br); t0=np.quantile(p[gte==0],br)
        yp=np.where(gte==1,(p>=t1),(p>=t0)).astype(int)
    return economics(yte,yp,gte)

SETS=[("german",load_german),("taiwan",load_taiwan),("gmsc",load_gmsc)]
SEEDS=list(range(30)); rows=[]
for name,loader in SETS:
    X,y,g,subs=loader()
    for cond in ["blind","gov_engine"]:
        arr=np.array([run(X,y,g,s,cond,subs) for s in SEEDS])
        m=arr.mean(0); ci=1.96*arr.std(0)/np.sqrt(len(SEEDS))
        rows.append({"dataset":name,"condition":cond,
                     "approval_overall":m[0],"approval_disadv":m[1],"approval_ref":m[2],
                     "approval_gap":m[2]-m[1],"profit":m[3],"profit_ci":ci[3]})
res=pd.DataFrame(rows)
res.to_csv(os.path.join(TAB_DIR,"exp7_economics.csv"),index=False)
print(res.round(4).to_string(index=False))

# Figure: profit (bars) and approval gap, blind vs governed, per dataset
sets=["german","taiwan","gmsc"]
fig,axes=plt.subplots(1,2,figsize=(10,4.2))
for ax,metric,lab in [(axes[0],"profit","Normalized profit per applicant"),
                      (axes[1],"approval_gap","Approval-rate gap (ref - disadvantaged)")]:
    bl=[res[(res.dataset==d)&(res.condition=='blind')][metric].values[0] for d in sets]
    gv=[res[(res.dataset==d)&(res.condition=='gov_engine')][metric].values[0] for d in sets]
    x=np.arange(len(sets))
    ax.bar(x-0.2,bl,0.4,label="blind",color=GRAYS[0],edgecolor="black",linewidth=1.0)
    ax.bar(x+0.2,gv,0.4,label="governed",color=GRAYS[2],edgecolor="black",linewidth=1.0)
    ax.axhline(0,color="black",linewidth=0.6)
    ax.set_xticks(x); ax.set_xticklabels(sets); ax.set_ylabel(lab); ax.legend(frameon=True,edgecolor="black")
fig.tight_layout(); save_fig(fig,"exp7_approval_profit"); plt.close(fig)
print("Done: economic impact (approval volume + profit) across 3 credit datasets.")


# ---- r/l profit sensitivity on German (where parity breaks) ----
def _profit_rl(yt,yp,r,l):
    ap=(yp==0); good=(ap&(yt==0)).sum(); bad=(ap&(yt==1)).sum()
    return (good*r-bad*l)/len(yt)
def _decisions(X,y,g,seed,gov):
    from sklearn.model_selection import train_test_split
    Xtr,Xte,ytr,yte,gtr,gte=train_test_split(X,y,g,test_size=0.3,random_state=seed,stratify=y)
    sc=StandardScaler().fit(Xtr); m=LogisticRegression(max_iter=2000).fit(sc.transform(Xtr),ytr)
    p=m.predict_proba(sc.transform(Xte))[:,1]
    if not gov: yp=(p>=0.5).astype(int)
    else:
        br=((p>=0.5).astype(int)==0).mean(); t1=np.quantile(p[gte==1],br); t0=np.quantile(p[gte==0],br)
        yp=np.where(gte==1,(p>=t1),(p>=t0)).astype(int)
    return yte,yp
Xg,yg,gg,_=load_german()
rl_rows=[]
for r,l in [(0.15,1.0),(0.25,1.0),(0.35,1.0),(0.25,0.5),(0.25,1.5)]:
    pb=np.mean([_profit_rl(*_decisions(Xg,yg,gg,s,False),r,l) for s in range(30)])
    pg=np.mean([_profit_rl(*_decisions(Xg,yg,gg,s,True),r,l) for s in range(30)])
    rl_rows.append({"r":r,"l":l,"profit_blind":pb,"profit_gov":pg,"delta":pg-pb})
rl=pd.DataFrame(rl_rows)
rl.to_csv(os.path.join(TAB_DIR,"exp7_rl_sensitivity.csv"),index=False)
print(rl.round(4).to_string(index=False))
print("Done: r/l profit sensitivity (German).")


dataset  condition  approval_overall  approval_disadv  approval_ref  approval_gap  profit  profit_ci
 german      blind            0.7597           0.6503        0.7788        0.1285 -0.0030     0.0047
 german gov_engine            0.7578           0.7528        0.7586        0.0059 -0.0054     0.0049
 taiwan      blind            0.9269           0.9321        0.9190       -0.0132  0.0201     0.0007
 taiwan gov_engine            0.9268           0.9268        0.9268       -0.0000  0.0201     0.0007
   gmsc      blind            0.9957           0.9915        0.9969        0.0054  0.1690     0.0007
   gmsc gov_engine            0.9955           0.9949        0.9956        0.0007  0.1690     0.0007


Done: economic impact (approval volume + profit) across 3 credit datasets.


   r   l  profit_blind  profit_gov   delta
0.15 1.0       -0.0635     -0.0656 -0.0021
0.25 1.0       -0.0030     -0.0054 -0.0024
0.35 1.0        0.0575      0.0548 -0.0028
0.25 0.5        0.0742      0.0725 -0.0016
0.25 1.5       -0.0802     -0.0834 -0.0032
Done: r/l profit sensitivity (German).
